# CelebA Data Exploration

This notebook explores the CelebA dataset for the gender recognition task.
It covers class distribution, sample images, and image statistics.

**Usage**: Set `DATA_ROOT` below to the path containing the CelebA files.

In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

DATA_ROOT = Path("../data/celeba")

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load annotations and partitions

In [ ]:
attrs = pd.read_csv(DATA_ROOT / "list_attr_celeba.txt", sep=r"\s+", header=1, index_col=0)
attrs["gender"] = attrs["Male"].map({1: "Male", -1: "Female"})

partitions = pd.read_csv(
    DATA_ROOT / "list_eval_partition.txt",
    sep=r"\s+", header=None, names=["filename", "partition"], index_col=0
)

df = attrs.join(partitions, how="inner")
split_labels = {0: "train", 1: "val", 2: "test"}
df["split"] = df["partition"].map(split_labels)

print(f"Total images: {len(df)}")
df.head()

## 2. Class distribution (overall and per split)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
df["gender"].value_counts().plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Overall Gender Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Per-split distribution
split_gender = df.groupby(["split", "gender"]).size().unstack(fill_value=0)
split_gender.plot.bar(ax=axes[1], color=["#DD8452", "#4C72B0"])
axes[1].set_title("Gender Distribution per Split")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Gender")

plt.tight_layout()
plt.savefig("../outputs/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPer-split counts:")
print(split_gender)

## 3. Sample images per class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for row, gender in enumerate(["Male", "Female"]):
    samples = df[df["gender"] == gender].sample(5, random_state=42)
    for col, (fname, _) in enumerate(samples.iterrows()):
        img = Image.open(DATA_ROOT / "img_align_celeba" / fname)
        axes[row, col].imshow(img)
        axes[row, col].set_title(gender, fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Sample Images per Class", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Image dimensions

In [ ]:
sample_files = df.sample(500, random_state=42).index.tolist()
sizes = []
for fname in sample_files:
    img = Image.open(DATA_ROOT / "img_align_celeba" / fname)
    sizes.append(img.size)  # (width, height)

widths, heights = zip(*sizes)
print(f"Width  — min: {min(widths)}, max: {max(widths)}, unique: {len(set(widths))}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, unique: {len(set(heights))}")

## 5. Summary statistics for the report

Collect key numbers to reference in the Data section of the final report.

In [ ]:
total = len(df)
male_pct = (df["gender"] == "Male").mean() * 100
female_pct = 100 - male_pct

print(f"Total images: {total:,}")
print(f"Male:   {male_pct:.1f}%")
print(f"Female: {female_pct:.1f}%")
print(f"\nSplit sizes:")
for split_name in ["train", "val", "test"]:
    n = (df["split"] == split_name).sum()
    print(f"  {split_name}: {n:,}")